# YOLOv8 Training — Football Ball Detection

This notebook trains a YOLOv8 model on your Roboflow-exported football ball detection dataset stored on Google Drive.

**Dataset structure expected (Roboflow format):**
```
football-ball-detection/
├── data.yaml
├── README.dataset.txt
├── README.roboflow.txt
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

**Steps:** Mount Drive → Install Ultralytics → Fix `data.yaml` paths → Train (50 epochs) → Validate → Run inference → Save results back to Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Set dataset path

Update `DATASET_DIR` below to point to the folder in your Drive that contains `data.yaml`, `train/`, `valid/`, `test/` (the folder shown in your screenshot).

In [ ]:
import os

# CHANGE THIS to the exact path of your dataset folder in Google Drive
DATASET_DIR = "/content/drive/MyDrive/football-ball-detection"

assert os.path.exists(DATASET_DIR), f"Dataset folder not found: {DATASET_DIR}. Update DATASET_DIR."
print("Contents of dataset folder:")
print(os.listdir(DATASET_DIR))


## 3. Install Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()


## 4. Inspect and fix `data.yaml`

Roboflow's `data.yaml` often uses relative paths (`../train/images`) which don't resolve well when run from a different working directory in Colab. This cell reads the existing `data.yaml`, prints it, and rewrites the `train` / `val` / `test` keys as absolute paths pointing to your Drive dataset, while keeping the original class names/count.

In [ ]:
import yaml

data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(data_cfg)

# Rewrite paths to absolute, matching the images subfolders shown in your dataset
data_cfg["train"] = os.path.join(DATASET_DIR, "train", "images")
data_cfg["val"]   = os.path.join(DATASET_DIR, "valid", "images")
data_cfg["test"]  = os.path.join(DATASET_DIR, "test", "images")

# Save a fixed copy (kept separate so your original data.yaml is untouched)
fixed_yaml_path = "/content/data_fixed.yaml"
with open(fixed_yaml_path, "w") as f:
    yaml.dump(data_cfg, f)

print("\nFixed data.yaml:")
print(data_cfg)


## 5. Sanity check: images/labels counts

Quick check that each split has matching numbers of images and label files, so training doesn't silently run on an empty or broken split.

In [ ]:
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split:6s} -> images: {n_img:5d} | labels: {n_lbl:5d}")


## 6. Train YOLOv8

Using `yolov8n.pt` (nano) as the starting checkpoint for speed — swap for `yolov8s.pt` / `yolov8m.pt` if you want higher accuracy and have the GPU budget. Training runs for **50 epochs** as requested.

Make sure your Colab runtime is set to a GPU: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 model as the starting point
model = YOLO("yolov8n.pt")

results = model.train(
    data=fixed_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,          # early stopping patience (won't stop before epoch 50 unless no improvement)
    project="/content/runs",
    name="football_ball_yolov8",
    exist_ok=True,
)


## 7. Validate the trained model

In [ ]:
metrics = model.val(data=fixed_yaml_path, split="val")
print(metrics)


## 8. Plot training curves

Displays the results.png (loss/mAP curves) and confusion matrix generated by Ultralytics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = "/content/runs/football_ball_yolov8"

for fname in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        img = mpimg.imread(fpath)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(fname)
        plt.show()
    else:
        print(f"Not found (yet): {fpath}")


## 9. Run inference on test images

Runs the best trained weights on the test split and saves annotated predictions.

In [ ]:
best_weights = os.path.join(run_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

test_img_dir = os.path.join(DATASET_DIR, "test", "images")

pred_results = trained_model.predict(
    source=test_img_dir,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/runs",
    name="football_ball_predictions",
    exist_ok=True,
)

print("Predictions saved to /content/runs/football_ball_predictions")


## 10. Preview a few predictions

In [ ]:
import glob

pred_dir = "/content/runs/football_ball_predictions"
pred_images = sorted(glob.glob(os.path.join(pred_dir, "*.jpg")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), pred_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(os.path.basename(img_path), fontsize=8)
plt.tight_layout()
plt.show()


## 11. Save trained weights back to Google Drive

Copies `best.pt` (and the whole run folder) to your Drive so results persist after the Colab session ends.

In [ ]:
import shutil

save_dir = os.path.join(DATASET_DIR, "yolov8_training_output")
os.makedirs(save_dir, exist_ok=True)

# Copy best/last weights
shutil.copy(os.path.join(run_dir, "weights", "best.pt"), os.path.join(save_dir, "best.pt"))
shutil.copy(os.path.join(run_dir, "weights", "last.pt"), os.path.join(save_dir, "last.pt"))

# Copy full run folder (metrics, plots, etc.)
run_backup = os.path.join(save_dir, "football_ball_yolov8_run")
if os.path.exists(run_backup):
    shutil.rmtree(run_backup)
shutil.copytree(run_dir, run_backup)

print(f"Saved weights and run artifacts to: {save_dir}")
